# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print("Dataset Title: ", metadata.name)
print("Dataset Description: ", metadata.description)

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all record sets and their fields by @id
record_sets = list(metadata.record_sets)
print(f"{len(record_sets)} record sets found in the dataset.")
for rs in record_sets:
    print(f"\nRecordSet name: {rs.name}")
    print(f"RecordSet @id: {rs.id}")
    print("Fields in this RecordSet:")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field.id}, dataType: {field.data_type})")

# Optionally print the first few records in the first record set as example
if record_sets:
    print(f"\nFirst 3 records in RecordSet '{record_sets[0].name}' (@id: {record_sets[0].id}):")
    count = 0
    for rec in dataset.records(record_set=record_sets[0].id):
        print(rec)
        count += 1
        if count >= 3:
            break

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
dataframes = {}
print("\nExtracting all record sets...")
for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    df = pd.DataFrame(records)
    dataframes[rs.id] = df
    print(f"RecordSet '{rs.name}' (@id: {rs.id}) loaded: {df.shape[0]} records, columns: {list(df.columns)}")

# For demonstration, pick the first record set as the focus for detailed EDA
main_record_set = record_sets[0] if record_sets else None

if main_record_set:
    print(f"\nData sample for '{main_record_set.name}' (@id: {main_record_set.id}):")
    display(dataframes[main_record_set.id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Automatically pick a numeric field for analysis (e.g. "MW" (molecular weight))
import numpy as np

if main_record_set:
    df = dataframes[main_record_set.id]
    # List possible numeric fields
    numeric_fields = [f.id for f in main_record_set.fields if f.data_type in ['schema:Number', 'schema:Float', 'schema:Integer']]
    print(f"Numeric fields in '{main_record_set.name}': {numeric_fields}")

    if numeric_fields:
        numeric_field = numeric_fields[0]  # e.g. 'MW' (molecular weight) or whatever is present

        # Some Croissant datasets may have string representations, so try to convert
        df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
        threshold = df[numeric_field].quantile(0.75)  # Use upper quartile as threshold

        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold} (upper quartile): {filtered_df.shape[0]} rows")

        # Normalization
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} (first five):")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        # Grouping by another suitable field, e.g. 'Sample' or 'accession'
        # Find a string field with not too high cardinality
        string_fields = [f.id for f in main_record_set.fields if f.data_type == 'schema:Text']
        group_field = None
        if string_fields:
            for sfield in string_fields:
                nunique = df[sfield].nunique()
                if 1 < nunique < df.shape[0] // 2:
                    group_field = sfield
                    break
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped by '{group_field}', mean of {numeric_field}:")
            display(grouped_df.head())
        else:
            print("No suitable string field found for grouping.")
    else:
        print("No numeric fields found in this record set.")
else:
    print("No record set to demonstrate EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
# Example visualization: Histogram and boxplot of the numeric field
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set and numeric_fields:
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(df[numeric_field].dropna(), bins=30, ax=axs[0], kde=True)
    axs[0].set_title(f"Distribution of {numeric_field}")
    sns.boxplot(x=df[numeric_field].dropna(), ax=axs[1])
    axs[1].set_title(f"Boxplot of {numeric_field}")
    plt.tight_layout()
    plt.show()

    # If a group_field exists, show bar plot of mean values
    if group_field is not None:
        plt.figure(figsize=(10, 5))
        order = grouped_df.sort_values(numeric_field, ascending=False)[group_field][:10]  # Top 10
        sns.barplot(data=grouped_df[grouped_df[group_field].isin(order)], x=group_field, y=numeric_field)
        plt.title(f"Mean {numeric_field} by {group_field} (Top 10 if many)")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Using `mlcroissant`, we loaded, explored, and analyzed the FAIR²-compliant dataset for mass spectrometry analysis of extracellular vesicles from human mast cells. The step-by-step data loading, overview, extraction, and EDA identified core numeric characteristics, normalized measurements, and group-level summaries, providing a strong foundation for further research, biomarker discovery, or advanced machine learning tasks.

For reproducible, interoperable AI research, always reference fields and record sets by `@id` as shown.